In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

huanranye_inria_aerial_image_labeling_dataset_path = kagglehub.dataset_download('huanranye/inria-aerial-image-labeling-dataset')
pravallika416_all_models_for_building_footprint_extraction_pytorch_default_1_path = kagglehub.model_download('pravallika416/all-models-for-building-footprint-extraction/PyTorch/default/1')

print('Data source import complete.')


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("huanranye/inria-aerial-image-labeling-dataset")

print("Path to dataset files:", path)

In [ ]:
!pip install segmentation-models-pytorch
!pip install segment-anything
!pip install timm
!pip install opencv-python

In [ ]:
import torch
import os
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

from torch.utils.data import Dataset,DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib

BASE="/kaggle/working"

IMG_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"

MASK_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:",device)

EPOCHS=100
BATCH=4
SIZE=512

In [ ]:
images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])

image_paths = [os.path.join(IMG_DIR, f) for f in images]
mask_paths = [os.path.join(MASK_DIR, f) for f in images]

print("Total:", len(image_paths))

In [ ]:
from sklearn.model_selection import train_test_split
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.1, random_state=42
)

In [ ]:
class InriaDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # IMAGE
        with rasterio.open(self.imgs[idx]) as src:
            img = src.read()  # (C,H,W)
            img = np.transpose(img, (1,2,0))

        img = img / 255.0
        img = np.transpose(img, (2,0,1))

        # MASK (single channel)
        with rasterio.open(self.masks[idx]) as src:
            mask = src.read(1)

        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)

In [ ]:
train_loader = DataLoader(
    InriaDataset(train_imgs, train_masks),
    batch_size=4,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    InriaDataset(val_imgs, val_masks),
    batch_size=4
)

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b4",   # stable & fast
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(device)

In [ ]:
class DiceLoss(torch.nn.Module):
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (pred * target).sum()
        return 1 - (2 * intersection + 1) / (pred.sum() + target.sum() + 1)

bce = torch.nn.BCEWithLogitsLoss()

def loss_fn(pred, target):
    return 0.5 * bce(pred, target) + 0.5 * DiceLoss()(pred, target)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import rasterio

best_loss = 1e9

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for img, mask in train_loader:
        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)
        loss = loss_fn(pred, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

    # SAVE BEST MODEL
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), BASE + "/unet_best.pth")
        print("✅ Best model saved")

In [ ]:
def evaluate(model):
    model.eval()
    TP = FP = FN = TN = 0

    with torch.no_grad():
        for img, mask in val_loader:
            img = img.to(device)
            mask = mask.to(device)

            pred = torch.sigmoid(model(img))
            pred = (pred > 0.5).float()

            TP += ((pred==1)&(mask==1)).sum().item()
            FP += ((pred==1)&(mask==0)).sum().item()
            FN += ((pred==0)&(mask==1)).sum().item()
            TN += ((pred==0)&(mask==0)).sum().item()

    iou = TP / (TP + FP + FN + 1e-6)
    dice = (2 * TP) / (2 * TP + FP + FN + 1e-6)
    accuracy = (TP + TN) / (TP + TN + FP + FN + 1e-6)

    return {
        "IoU": iou,
        "Dice": dice,
        "Accuracy": accuracy
    }

In [ ]:
model.load_state_dict(torch.load(BASE + "/unet_best.pth"))

metrics = evaluate(model)
print("FINAL UNET:", metrics)

In [ ]:
import os
import json
import torch
import shutil
import matplotlib.pyplot as plt
from IPython.display import FileLink

# ================== FOLDER ==================
SAVE_DIR = "/kaggle/working/unet_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================== SAVE METRICS ==================
with open(SAVE_DIR + "/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# ================== SAVE MODEL ==================
torch.save(model.state_dict(), SAVE_DIR + "/unet_final.pth")

# ================== SAVE PREDICTIONS (5 SAMPLES) ==================
model.eval()
count = 0

with torch.no_grad():
    for img, mask in val_loader:
        img = img.to(device)
        pred = model(img)
        pred = torch.sigmoid(pred).cpu()

        for i in range(img.size(0)):
            if count == 5:
                break

            plt.figure(figsize=(10,3))

            plt.subplot(1,3,1)
            plt.title("Image")
            plt.imshow(img[i].cpu().permute(1,2,0))

            plt.subplot(1,3,2)
            plt.title("Ground Truth")
            plt.imshow(mask[i].cpu().squeeze(), cmap="gray")

            plt.subplot(1,3,3)
            plt.title("Prediction")
            plt.imshow(pred[i].squeeze(), cmap="gray")

            plt.savefig(f"{SAVE_DIR}/pred_{count}.png")
            plt.close()

            count += 1

        if count == 5:
            break

# ================== SAVE CURVES ==================
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss Curve")
plt.savefig(SAVE_DIR + "/loss_curve.png")
plt.close()

plt.figure()
plt.plot(train_acc, label="Train Accuracy")
plt.plot(val_acc, label="Val Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.savefig(SAVE_DIR + "/accuracy_curve.png")
plt.close()

# ================== ZIP FOLDER ==================
zip_path = "/kaggle/working/unet_results.zip"
shutil.make_archive("/kaggle/working/unet_results", 'zip', SAVE_DIR)

print("Saved + Zipped successfully!")

# ================== DOWNLOAD LINK ==================
FileLink(zip_path)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import rasterio
import torch.nn.functional as F
import segmentation_models_pytorch as smp

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD MODEL =================
MODEL_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
)

state_dict = torch.load(MODEL_PATH, map_location="cpu")
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("✅ Model loaded successfully")

# ================= METRICS =================
def dice_score(pred, target):
    smooth = 1e-6
    pred = (pred > 0.5).float()
    return (2 * (pred * target).sum() + smooth) / (pred.sum() + target.sum() + smooth)

def iou_score(pred, target):
    smooth = 1e-6
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + smooth) / (union + smooth)

# ================= SAVE DIR =================
SAVE_DIR = "/kaggle/working/final_results2"
os.makedirs(SAVE_DIR, exist_ok=True)

dice_list, iou_list = [], []

# ================= INFERENCE (SAFE) =================
model.eval()

for i, (img, mask) in enumerate(val_loader):

    if i >= 5:
        break

    img = img[:1]
    mask = mask[:1]

    # 🔥 IMPORTANT: resize to avoid OOM
    img = F.interpolate(img, size=(256, 256), mode="bilinear", align_corners=False)
    mask = F.interpolate(mask, size=(256, 256), mode="nearest")

    img = img.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        pred = model(img)
        pred = torch.sigmoid(pred)

    dice_list.append(dice_score(pred, mask).item())
    iou_list.append(iou_score(pred, mask).item())

    # ================= PLOT =================
    img_np = img[0].cpu().permute(1, 2, 0).numpy()
    mask_np = mask[0].cpu().squeeze().numpy()
    pred_np = (pred[0].cpu().squeeze().numpy() > 0.5)

    plt.figure(figsize=(10, 3))

    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title("Image")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_np, cmap="gray")
    plt.title("Ground Truth")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_np, cmap="gray")
    plt.title("Prediction")

    plt.tight_layout()
    plt.show()

# ================= FINAL METRICS =================
metrics = {
    "mean_dice": float(np.mean(dice_list)),
    "mean_iou": float(np.mean(iou_list))
}

print("📊 Final Metrics:", metrics)

with open(SAVE_DIR + "/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

np.save(SAVE_DIR + "/dice.npy", dice_list)
np.save(SAVE_DIR + "/iou.npy", iou_list)

print("✅ Saved in:", SAVE_DIR)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import rasterio
import torch.nn.functional as F
import segmentation_models_pytorch as smp

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD MODEL =================
MODEL_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
)

state_dict = torch.load(MODEL_PATH, map_location="cpu")
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("✅ Model loaded successfully")

# ================= METRICS =================
def dice_score(pred, target):
    smooth = 1e-6
    pred = (pred > 0.5).float()
    return (2 * (pred * target).sum() + smooth) / (pred.sum() + target.sum() + smooth)

def iou_score(pred, target):
    smooth = 1e-6
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + smooth) / (union + smooth)

# ================= SAVE DIR =================
SAVE_DIR = "/kaggle/working/final_results2"
os.makedirs(SAVE_DIR, exist_ok=True)

dice_list, iou_list = [], []

# ================= INFERENCE =================
for i, (img, mask) in enumerate(val_loader):

    if i >= 5:
        break

    img = img[:1]
    mask = mask[:1]

    img = F.interpolate(img, size=(256, 256), mode="bilinear", align_corners=False)
    mask = F.interpolate(mask, size=(256, 256), mode="nearest")

    img = img.to(device)
    mask = mask.to(device)

    with torch.no_grad():
        pred = model(img)
        pred = torch.sigmoid(pred)

    dice_list.append(dice_score(pred, mask).item())
    iou_list.append(iou_score(pred, mask).item())

    # ================= NUMPY =================
    img_np = img[0].cpu().permute(1, 2, 0).numpy()
    mask_np = mask[0].cpu().squeeze().numpy()
    pred_np = (pred[0].cpu().squeeze().numpy() > 0.5).astype(np.uint8)

    # ================= OVERLAY =================
    overlay = img_np.copy()

    # RED MASK (building prediction)
    overlay[pred_np == 1] = [1, 0, 0]   # red

    # blend original + overlay
    blended = (0.7 * img_np + 0.3 * overlay)

    # ================= PLOT =================
    plt.figure(figsize=(12, 3))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.title("Image")

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap="gray")
    plt.title("Ground Truth")

    plt.subplot(1, 4, 3)
    plt.imshow(pred_np, cmap="gray")
    plt.title("Prediction")

    plt.subplot(1, 4, 4)
    plt.imshow(blended)
    plt.title("Red Overlay")

    plt.tight_layout()
    plt.show()

# ================= FINAL METRICS =================
metrics = {
    "mean_dice": float(np.mean(dice_list)),
    "mean_iou": float(np.mean(iou_list))
}

print("📊 Final Metrics:", metrics)

with open(SAVE_DIR + "/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

np.save(SAVE_DIR + "/dice.npy", dice_list)
np.save(SAVE_DIR + "/iou.npy", iou_list)

print("✅ Saved in:", SAVE_DIR)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import os

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= MODEL PATHS =================
MODEL_PATHS = {
    "UNet": "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth",
    "DeepLab": "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth",
    "RF": "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/rf_model.pkl",      # (if sklearn model saved as torch/np)
    "SVM": "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/svm_model.pkl",
    "SAM": "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/sam_model.pth"
}

# ================= LOAD SEGMENTATION MODELS =================
def load_smp_model(model_type, path):
    if model_type == "UNet":
        model = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1)
    elif model_type == "DeepLab":
        model = smp.DeepLabV3Plus("resnet34", encoder_weights=None, in_channels=3, classes=1)
    else:
        # placeholder for RF/SVM/SAM wrappers
        model = None

    if model is not None:
        state = torch.load(path, map_location="cpu")
        state = {k.replace("module.", ""): v for k, v in state.items()}
        model.load_state_dict(state)
        model.to(device)
        model.eval()

    return model

models = {name: load_smp_model(name, path) for name, path in MODEL_PATHS.items()}

# ================= PREPROCESS =================
def preprocess(img):
    img = F.interpolate(img, size=(256, 256), mode="bilinear", align_corners=False)
    return img

# ================= PREDICT FUNCTION =================
def predict(model, img):
    if model is None:
        # fake placeholder for RF/SVM/SAM (replace with real logic)
        return torch.zeros((1, 1, 256, 256))

    with torch.no_grad():
        out = model(img)
        out = torch.sigmoid(out)
    return out

# ================= VISUALIZATION =================
num_samples = 5

for i, (img, mask) in enumerate(val_loader):

    if i >= num_samples:
        break

    img = img[:1].to(device)
    mask = mask[:1]

    img_resized = preprocess(img)

    preds = {}

    for name, model in models.items():
        preds[name] = predict(model, img_resized)[0,0].cpu().numpy()

    # ================= PLOT GRID =================
    fig, axes = plt.subplots(1, 2 + len(models), figsize=(18, 3))

    img_np = img[0].cpu().permute(1,2,0).numpy()
    mask_np = mask[0].squeeze().numpy()

    axes[0].imshow(img_np)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask_np, cmap="gray")
    axes[1].set_title("GT")
    axes[1].axis("off")

    idx = 2
    for name in models.keys():
        pred_bin = (preds[name] > 0.5).astype(np.uint8)
        axes[idx].imshow(pred_bin, cmap="gray")
        axes[idx].set_title(name)
        axes[idx].axis("off")
        idx += 1

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import numpy as np
import cv2
import pickle
import matplotlib.pyplot as plt
from tifffile import imread

from segment_anything import sam_model_registry
from segment_anything import SamPredictor

device="cpu"

In [ ]:
import torch
import joblib
from segment_anything import sam_model_registry
from segment_anything import SamPredictor

device="cpu"

unet=torch.load(
"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth",
map_location=device)

deeplab=torch.load(
"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth",
map_location=device)

rf=joblib.load(
"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/rf_model.pkl"
)

sam=sam_model_registry["vit_b"](
checkpoint="/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/sam_model.pth"
)

predictor=SamPredictor(sam)

print("ALL MODELS LOADED SUCCESSFULLY")

In [ ]:
def preprocess(img):

    img=cv2.resize(img,(256,256))

    img=img/255.0

    tensor=np.transpose(img,(2,0,1))

    tensor=torch.tensor(
        tensor).float().unsqueeze(0)

    return tensor,img

In [ ]:
def cnn_predict(model,img):

    tensor,img=preprocess(img)

    with torch.no_grad():

        pred=model(tensor)

        pred=torch.sigmoid(pred)

        pred=pred.squeeze().numpy()

    mask=(pred>0.5).astype(np.uint8)

    return mask,pred

In [ ]:
def unet_deeplab(img):

    umask,uprob=cnn_predict(unet,img)

    dmask,dprob=cnn_predict(deeplab,img)

    fusion=(uprob+dprob)/2

    final=(fusion>0.5).astype(np.uint8)

    return final

In [ ]:
def sam_refine(img,mask):

    img=cv2.resize(img,(256,256))

    predictor.set_image(img)

    sam_masks,_,_=predictor.predict(
        point_coords=None,
        point_labels=None,
        multimask_output=False
    )

    sam_mask=sam_masks[0].astype(np.uint8)

    fusion=(mask+sam_mask)/2

    final=(fusion>0.5).astype(np.uint8)

    return final

In [ ]:
def unet_sam(img):

    umask,_=cnn_predict(unet,img)

    final=sam_refine(img,umask)

    return final

In [ ]:
def deeplab_sam(img):

    dmask,_=cnn_predict(deeplab,img)

    final=sam_refine(img,dmask)

    return final

In [ ]:
def rf_model(img):

    _,prob=cnn_predict(unet,img)

    features=prob.reshape(-1,1)

    pred=rf.predict(features)

    pred=pred.reshape(256,256)

    return pred

In [ ]:
!pip install segmentation-models-pytorch
!pip install segment-anything
!pip install timm
!pip install opencv-python

In [ ]:
import torch
import segmentation_models_pytorch as smp
import joblib

device="cpu"

###################################
# SAFE CHECKPOINT LOADER FUNCTION
###################################

def load_weights(path):

    ckpt=torch.load(path,map_location=device)

    if isinstance(ckpt,dict):

        if 'state_dict' in ckpt:
            return ckpt['state_dict']

        elif 'model' in ckpt:
            return ckpt['model']

        elif 'net' in ckpt:
            return ckpt['net']

        else:
            return ckpt

    else:
        return ckpt


###################################
# UNET
###################################

unet=smp.Unet(

    encoder_name="resnet34",

    encoder_weights=None,

    in_channels=3,

    classes=1

).to(device)

unet_weights=load_weights(

"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth"

)

unet.load_state_dict(unet_weights,strict=False)

unet.eval()


###################################
# DEEPLAB
###################################

deeplab=smp.DeepLabV3Plus(

    encoder_name="resnet34",

    encoder_weights=None,

    in_channels=3,

    classes=1

).to(device)

deeplab_weights=load_weights(

"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth"

)

deeplab.load_state_dict(deeplab_weights,strict=False)

deeplab.eval()


###################################
# RANDOM FOREST
###################################

rf=joblib.load(

"/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/rf_model.pkl"

)


###################################
# FINAL CHECK
###################################

print(type(unet))
print(type(deeplab))
print(type(rf))

print("ALL MODELS LOADED SUCCESSFULLY")

In [ ]:
import cv2
import numpy as np
import torch

def cnn_predict(model,img):

    img=cv2.resize(img,(256,256))

    tensor=torch.tensor(img).float()/255

    tensor=tensor.permute(2,0,1).unsqueeze(0)

    tensor=tensor.to("cpu")

    with torch.no_grad():

        pred=model(tensor)

        pred=torch.sigmoid(pred)

        pred=pred.cpu().numpy()[0,0]

    mask=(pred>0.5).astype(np.uint8)

    return mask,pred


def rf_model(img):

    img=cv2.resize(img,(256,256))

    gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)

    edges=cv2.Canny(gray,50,150)

    # normalize
    r=img[:,:,0]/255
    g=img[:,:,1]/255
    b=img[:,:,2]/255
    gray=gray/255
    edges=edges/255

    features=np.stack([r,g,b,gray,edges],axis=2)

    feat=features.reshape(-1,5)

    pred=rf.predict(feat)

    mask=pred.reshape(256,256)

    return mask.astype(np.uint8)

In [ ]:
def unet_deeplab(img):

    umask,_=cnn_predict(unet,img)

    dmask,_=cnn_predict(deeplab,img)

    hybrid=((umask+dmask)>=1).astype(np.uint8)

    return hybrid


def unet_sam(img):

    umask,_=cnn_predict(unet,img)

    return umask


def deeplab_sam(img):

    dmask,_=cnn_predict(deeplab,img)

    return dmask

In [ ]:
def metrics(gt,pred):

    tp=np.sum((gt==1)&(pred==1))

    tn=np.sum((gt==0)&(pred==0))

    fp=np.sum((gt==0)&(pred==1))

    fn=np.sum((gt==1)&(pred==0))


    acc=(tp+tn)/(tp+tn+fp+fn+1e-6)

    iou=tp/(tp+fp+fn+1e-6)

    dice=(2*tp)/(2*tp+fp+fn+1e-6)

    prec=tp/(tp+fp+1e-6)

    rec=tp/(tp+fn+1e-6)

    return acc,iou,dice,prec,rec

In [ ]:
import os
import random
import matplotlib.pyplot as plt

IMG_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"

GT_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"


files=[f for f in os.listdir(IMG_DIR) if f.endswith(".tif")]

samples=random.sample(files,5)

print("Selected:",samples)


def overlay(img,mask):

    o=img.copy()

    o[mask==1]=[255,0,0]

    return o


results=[]


for file in samples:

    print("\nProcessing:",file)

    img=cv2.imread(IMG_DIR+"/"+file)

    img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    gt=cv2.imread(GT_DIR+"/"+file,0)

    img=cv2.resize(img,(256,256))

    gt=cv2.resize(gt,(256,256))

    gt=(gt>0).astype(np.uint8)


    ud=unet_deeplab(img)

    us=unet_sam(img)

    ds=deeplab_sam(img)

    rfmask=rf_model(img)


    results.append([

        metrics(gt,ud),

        metrics(gt,us),

        metrics(gt,ds),

        metrics(gt,rfmask)

    ])


    plt.figure(figsize=(18,10))

    plt.subplot(251)
    plt.title("Original")
    plt.imshow(img)
    plt.axis("off")

    plt.subplot(252)
    plt.title("GT")
    plt.imshow(gt,cmap="gray")
    plt.axis("off")

    plt.subplot(253)
    plt.title("UNet+DeepLab")
    plt.imshow(ud,cmap="gray")
    plt.axis("off")

    plt.subplot(254)
    plt.title("UNet")
    plt.imshow(us,cmap="gray")
    plt.axis("off")

    plt.subplot(255)
    plt.title("DeepLab")
    plt.imshow(ds,cmap="gray")
    plt.axis("off")


    plt.subplot(256)
    plt.title("GT Overlay")
    plt.imshow(overlay(img,gt))
    plt.axis("off")

    plt.subplot(257)
    plt.title("UD Overlay")
    plt.imshow(overlay(img,ud))
    plt.axis("off")

    plt.subplot(258)
    plt.title("UNet Overlay")
    plt.imshow(overlay(img,us))
    plt.axis("off")

    plt.subplot(259)
    plt.title("DeepLab Overlay")
    plt.imshow(overlay(img,ds))
    plt.axis("off")

    plt.subplot(2,5,10)
    plt.title("RF Overlay")
    plt.imshow(overlay(img,rfmask))
    plt.axis("off")

    plt.tight_layout()

    plt.show()

In [ ]:
results=np.array(results)

avg=np.mean(results,axis=0)

names=["UNet+DeepLab","UNet","DeepLab","RF"]

print("\nAVERAGE RESULTS")

for i,name in enumerate(names):

    print("\n",name)

    print("Accuracy:",avg[i][0])

    print("IoU:",avg[i][1])

    print("Dice:",avg[i][2])

    print("Precision:",avg[i][3])

    print("Recall:",avg[i][4])

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(names,avg[:,1])

plt.title("IoU Comparison")

plt.ylabel("IoU")

plt.show()

In [ ]:
!pip install segmentation-models-pytorch
!pip install albumentations
!pip install torchmetrics

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

from torch.utils.data import Dataset,DataLoader

import albumentations as A

from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
IMG_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"

GT_DIR="/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

In [ ]:
train_aug=A.Compose([

A.Resize(256,256),

A.HorizontalFlip(p=0.5),

A.VerticalFlip(p=0.5),

A.RandomRotate90(p=0.5),

A.RandomBrightnessContrast(p=0.3),

A.GaussianBlur(p=0.2),

A.Normalize()

])

val_aug=A.Compose([

A.Resize(256,256),

A.Normalize()

])

In [ ]:
class BuildingDataset(Dataset):

    def __init__(self,img_dir,gt_dir,files,aug):

        self.img_dir=img_dir

        self.gt_dir=gt_dir

        self.files=files

        self.aug=aug


    def __len__(self):

        return len(self.files)


    def __getitem__(self,idx):

        file=self.files[idx]

        img=cv2.imread(self.img_dir+"/"+file)

        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        mask=cv2.imread(self.gt_dir+"/"+file,0)

        mask=(mask>0).astype(np.float32)

        augmented=self.aug(image=img,mask=mask)

        img=augmented['image']

        mask=augmented['mask']

        img=torch.tensor(img).permute(2,0,1).float()

        mask=torch.tensor(mask).unsqueeze(0).float()

        return img,mask

In [ ]:
files=[f for f in os.listdir(IMG_DIR) if f.endswith(".tif")]

np.random.shuffle(files)

split=int(len(files)*0.9)

train_files=files[:split]

val_files=files[split:]

print("Train:",len(train_files))

print("Val:",len(val_files))

In [ ]:
train_ds=BuildingDataset(IMG_DIR,GT_DIR,train_files,train_aug)

val_ds=BuildingDataset(IMG_DIR,GT_DIR,val_files,val_aug)

train_loader=DataLoader(

train_ds,

batch_size=8,

shuffle=True,

num_workers=2

)

val_loader=DataLoader(

val_ds,

batch_size=8,

shuffle=False

)

In [ ]:
device="cpu"

unet=smp.Unet(

encoder_name="resnet34",

encoder_weights="imagenet",

in_channels=3,

classes=1

).to(device)


deeplab=smp.DeepLabV3Plus(

encoder_name="resnet34",

encoder_weights="imagenet",

in_channels=3,

classes=1

).to(device)

In [ ]:
dice=smp.losses.DiceLoss(

mode='binary'

)

bce=nn.BCEWithLogitsLoss()

focal=smp.losses.FocalLoss(

mode='binary'

)


def loss_fn(pred,gt):

    return dice(pred,gt)+bce(pred,gt)+focal(pred,gt)

In [ ]:
unet_opt=optim.AdamW(

unet.parameters(),

lr=1e-4,

weight_decay=1e-5

)

deeplab_opt=optim.AdamW(

deeplab.parameters(),

lr=1e-4,

weight_decay=1e-5

)


unet_sched=optim.lr_scheduler.ReduceLROnPlateau(

unet_opt,

patience=3

)

deeplab_sched=optim.lr_scheduler.ReduceLROnPlateau(

deeplab_opt,

patience=3
)

In [ ]:
def train_epoch(model,loader,opt):

    model.train()

    total=0

    for img,mask in tqdm(loader):

        img=img.to(device)

        mask=mask.to(device)

        pred=model(img)

        loss=loss_fn(pred,mask)

        opt.zero_grad()

        loss.backward()

        opt.step()

        total+=loss.item()

    return total/len(loader)

In [ ]:
def val_epoch(model,loader):

    model.eval()

    total=0

    with torch.no_grad():

        for img,mask in loader:

            img=img.to(device)

            mask=mask.to(device)

            pred=model(img)

            loss=loss_fn(pred,mask)

            total+=loss.item()

    return total/len(loader)

In [ ]:
EPOCHS=35

for epoch in range(EPOCHS):

    print("\nEpoch",epoch)

    utrain=train_epoch(

        unet,

        train_loader,

        unet_opt

    )

    uval=val_epoch(

        unet,

        val_loader

    )

    dtrain=train_epoch(

        deeplab,

        train_loader,

        deeplab_opt

    )

    dval=val_epoch(

        deeplab,

        val_loader

    )


    unet_sched.step(uval)

    deeplab_sched.step(dval)


    print("UNET Train:",utrain)

    print("UNET Val:",uval)

    print("DEEPLAB Train:",dtrain)

    print("DEEPLAB Val:",dval)


    torch.save(

        unet.state_dict(),

        "unet_best.pth"

    )

    torch.save(

        deeplab.state_dict(),

        "deeplab_best.pth"

    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from torch.cuda.amp import autocast,GradScaler

EPOCHS=35
ACCUM_STEPS=2
PATIENCE=6

scaler=GradScaler(enabled=(device=='cuda'))

best_unet=float('inf')
best_deeplab=float('inf')

patience_u=0
patience_d=0

unet_train_hist=[]
unet_val_hist=[]

deeplab_train_hist=[]
deeplab_val_hist=[]

for epoch in range(EPOCHS):

    print(f"\n========== EPOCH {epoch+1}/{EPOCHS} ==========")

    ### UNET TRAIN ###
    unet.train()

    running=0

    unet_opt.zero_grad()

    for i,(img,mask) in enumerate(train_loader):

        img=img.to(device)
        mask=mask.to(device)

        with autocast(enabled=(device=='cuda')):

            pred=unet(img)

            loss=loss_fn(pred,mask)

            loss=loss/ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i+1)%ACCUM_STEPS==0:

            scaler.unscale_(unet_opt)

            torch.nn.utils.clip_grad_norm_(

                unet.parameters(),1.0

            )

            scaler.step(unet_opt)

            scaler.update()

            unet_opt.zero_grad()

        running+=loss.item()

    utrain=running/len(train_loader)


    ### UNET VAL ###
    unet.eval()

    running=0

    with torch.no_grad():

        for img,mask in val_loader:

            img=img.to(device)
            mask=mask.to(device)

            pred=unet(img)

            loss=loss_fn(pred,mask)

            running+=loss.item()

    uval=running/len(val_loader)


    ### DEEPLAB TRAIN ###
    deeplab.train()

    running=0

    deeplab_opt.zero_grad()

    for i,(img,mask) in enumerate(train_loader):

        img=img.to(device)
        mask=mask.to(device)

        with autocast(enabled=(device=='cuda')):

            pred=deeplab(img)

            loss=loss_fn(pred,mask)

            loss=loss/ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i+1)%ACCUM_STEPS==0:

            scaler.unscale_(deeplab_opt)

            torch.nn.utils.clip_grad_norm_(

                deeplab.parameters(),1.0

            )

            scaler.step(deeplab_opt)

            scaler.update()

            deeplab_opt.zero_grad()

        running+=loss.item()

    dtrain=running/len(train_loader)


    ### DEEPLAB VAL ###
    deeplab.eval()

    running=0

    with torch.no_grad():

        for img,mask in val_loader:

            img=img.to(device)
            mask=mask.to(device)

            pred=deeplab(img)

            loss=loss_fn(pred,mask)

            running+=loss.item()

    dval=running/len(val_loader)


    ### Scheduler ###
    unet_sched.step(uval)
    deeplab_sched.step(dval)


    ### History ###
    unet_train_hist.append(utrain)
    unet_val_hist.append(uval)

    deeplab_train_hist.append(dtrain)
    deeplab_val_hist.append(dval)


    ### PRINT ###
    print(f"UNET   Train : {utrain:.4f}")
    print(f"UNET   Val   : {uval:.4f}")

    print(f"DeepLab Train: {dtrain:.4f}")
    print(f"DeepLab Val  : {dval:.4f}")


    ### SAVE BEST ###
    if uval < best_unet:

        best_unet=uval

        torch.save(

            unet.state_dict(),

            "unet_best.pth"

        )

        patience_u=0

        print("UNET BEST SAVED")

    else:

        patience_u+=1


    if dval < best_deeplab:

        best_deeplab=dval

        torch.save(

            deeplab.state_dict(),

            "deeplab_best.pth"

        )

        patience_d=0

        print("DEEPLAB BEST SAVED")

    else:

        patience_d+=1


    ### EARLY STOP ###
    if patience_u>PATIENCE and patience_d>PATIENCE:

        print("EARLY STOPPING")

        break


print("\nTRAINING FINISHED")

In [ ]:
def edge_loss(pred,mask):

    sobelx=cv2.Sobel(

        mask.cpu().numpy()[0,0],

        cv2.CV_32F,

        1,0

    )

    sobely=cv2.Sobel(

        mask.cpu().numpy()[0,0],

        cv2.CV_32F,

        0,1

    )

    edge=np.sqrt(

        sobelx**2+sobely**2

    )

    edge=torch.tensor(edge).to(device)

    return torch.mean(

        (pred[0,0]-edge)**2

    )

In [ ]:
# ================= INSTALL (if needed) =================
!pip install rasterio geopandas shapely fiona
!pip install segmentation-models-pytorch
!pip install segment-anything
!pip install timm
!pip install opencv-python

# ================= IMPORTS =================
import os
import cv2
import torch
import numpy as np
import warnings
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")  # 🔥 removes TIFF warnings

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ================= DATASET =================
SIZE = 512

class TiffDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # 🔥 Use rasterio instead of cv2 (fix TIFF warnings + geo data safe)
        with rasterio.open(self.imgs[idx]) as src:
            img = src.read([1,2,3]).transpose(1,2,0)

        img = cv2.resize(img, (SIZE, SIZE))
        img = img / 255.0

        with rasterio.open(self.masks[idx]) as src:
            mask = src.read(1)

        mask = cv2.resize(mask, (SIZE, SIZE))
        mask = (mask > 0).astype(np.float32)

        img = np.transpose(img, (2,0,1))
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img, dtype=torch.float32), \
               torch.tensor(mask, dtype=torch.float32)

# ================= PATHS =================
IMG_DIR = "//kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])

image_paths = [os.path.join(IMG_DIR, f) for f in images]
mask_paths  = [os.path.join(MASK_DIR, f) for f in images]

# ================= SPLIT =================
split = int(0.8 * len(image_paths))
train_imgs = image_paths[:split]
val_imgs   = image_paths[split:]

train_masks = mask_paths[:split]
val_masks   = mask_paths[split:]

print("Train:", len(train_imgs), "Val:", len(val_imgs))

# ================= DATALOADER =================
BATCH = 4

train_loader = DataLoader(
    TiffDataset(train_imgs, train_masks),
    batch_size=BATCH,
    shuffle=True,
    num_workers=0,   # 🔥 stable for TIFF
    pin_memory=True
)

val_loader = DataLoader(
    TiffDataset(val_imgs, val_masks),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# ================= MODEL =================
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

model = torch.nn.DataParallel(model)
model = model.to(device)

# ================= LOSS =================
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

scaler = torch.amp.GradScaler("cuda")

# ================= RESUME =================
CHECKPOINT = "/kaggle/working/unet_resume.pth"
start_epoch = 0

if os.path.exists(CHECKPOINT):
    model.module.load_state_dict(torch.load(CHECKPOINT))
    print("✅ Resumed previous training")

# ================= TRAIN =================
EPOCHS = 50
best_val = 1e9

for epoch in range(start_epoch, EPOCHS):

    model.train()
    train_loss = 0

    for img, mask in train_loader:
        img = img.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            pred = model(img)
            loss = loss_fn(pred, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for img, mask in val_loader:
            img = img.to(device)
            mask = mask.to(device)

            with torch.amp.autocast("cuda"):
                pred = model(img)
                loss = loss_fn(pred, mask)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # ===== SAVE =====
    torch.save(model.module.state_dict(), CHECKPOINT)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.module.state_dict(), "/kaggle/working/unet_best.pth")
        print("✅ Best Saved")

# ================= PREDICTION + EXPORT =================
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.eval()

for i, path in enumerate(val_imgs[:5]):  # test few images

    with rasterio.open(path) as src:
        img = src.read([1,2,3]).transpose(1,2,0)
        transform = src.transform

    orig_size = img.shape[:2]

    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    img_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred = model(img_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (orig_size[1], orig_size[0]))
    mask_bin = (pred > 0.5).astype(np.uint8)

    # ===== SAVE JPG =====
    cv2.imwrite(f"{OUTPUT_DIR}/mask_{i}.jpg", mask_bin*255)

    # ===== SAVE TIFF =====
    with rasterio.open(
        f"{OUTPUT_DIR}/mask_{i}.tif",
        'w',
        driver='GTiff',
        height=mask_bin.shape[0],
        width=mask_bin.shape[1],
        count=1,
        dtype=mask_bin.dtype,
        transform=transform
    ) as dst:
        dst.write(mask_bin, 1)

    # ===== SHAPEFILE =====
    contours, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    for cnt in contours:
        if len(cnt) > 3:
            poly = Polygon(cnt.squeeze())
            polygons.append(poly)

    gdf = gpd.GeoDataFrame(geometry=polygons)
    gdf.to_file(f"{OUTPUT_DIR}/mask_{i}.shp")

print("🎯 DONE: TIFF + JPG + SHP generated!")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch
import numpy as np
import random
import rasterio

# ================= SETTINGS =================
NUM_SAMPLES = 5
SIZE = 512

model.eval()

# pick random samples from validation
indices = random.sample(range(len(val_imgs)), NUM_SAMPLES)

for i, idx in enumerate(indices):

    # ===== LOAD IMAGE =====
    with rasterio.open(val_imgs[idx]) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    with rasterio.open(val_masks[idx]) as src:
        gt_mask = src.read(1)

    orig_h, orig_w = img.shape[:2]

    # ===== PREPROCESS =====
    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    input_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    # ===== PREDICTION =====
    with torch.no_grad():
        pred = model(input_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (orig_w, orig_h))
    pred_mask = (pred > 0.5).astype(np.uint8)

    # ===== OVERLAY =====
    overlay = img.copy()
    overlay[pred_mask == 1] = [255, 0, 0]  # red buildings

    # ===== BOUNDARY =====
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_img = img.copy()
    cv2.drawContours(boundary_img, contours, -1, (0,255,0), 2)

    # ===== PLOT =====
    plt.figure(figsize=(15,5))

    plt.subplot(1,5,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,5,2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,5,3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

    plt.subplot(1,5,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.subplot(1,5,5)
    plt.imshow(boundary_img)
    plt.title("Boundary")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch
import numpy as np
import random
import rasterio

# ================= SETTINGS =================
NUM_SAMPLES = 5
SIZE = 512

model.eval()

# pick random samples from validation
indices = random.sample(range(len(val_imgs)), NUM_SAMPLES)

for i, idx in enumerate(indices):

    # ===== LOAD IMAGE =====
    with rasterio.open(val_imgs[idx]) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    with rasterio.open(val_masks[idx]) as src:
        gt_mask = src.read(1)

    orig_h, orig_w = img.shape[:2]

    # ===== PREPROCESS =====
    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    input_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    # ===== PREDICTION =====
    with torch.no_grad():
        pred = model(input_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (orig_w, orig_h))
    pred_mask = (pred > 0.5).astype(np.uint8)

    # ===== OVERLAY =====
    overlay = img.copy()
    overlay[pred_mask == 1] = [255, 0, 0]  # red buildings

    # ===== BOUNDARY =====
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_img = img.copy()
    cv2.drawContours(boundary_img, contours, -1, (0,255,0), 2)

    # ===== PLOT =====
    plt.figure(figsize=(15,5))

    plt.subplot(1,5,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,5,2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,5,3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

    plt.subplot(1,5,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.subplot(1,5,5)
    plt.imshow(boundary_img)
    plt.title("Boundary")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import numpy as np

model.eval()

def compute_metrics(pred, mask):
    pred = (pred > 0.5).astype(np.uint8)
    mask = (mask > 0.5).astype(np.uint8)

    TP = np.sum((pred == 1) & (mask == 1))
    TN = np.sum((pred == 0) & (mask == 0))
    FP = np.sum((pred == 1) & (mask == 0))
    FN = np.sum((pred == 0) & (mask == 1))

    # Avoid division errors
    eps = 1e-7

    iou = TP / (TP + FP + FN + eps)
    dice = (2 * TP) / (2 * TP + FP + FN + eps)
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)

    return iou, dice, precision, recall, f1

# ================= EVALUATION LOOP =================
iou_list, dice_list = [], []
prec_list, rec_list, f1_list = [], [], []

with torch.no_grad():
    for img, mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)
        pred = torch.sigmoid(pred).cpu().numpy()
        mask = mask.cpu().numpy()

        for p, m in zip(pred, mask):
            iou, dice, precision, recall, f1 = compute_metrics(p[0], m[0])

            iou_list.append(iou)
            dice_list.append(dice)
            prec_list.append(precision)
            rec_list.append(recall)
            f1_list.append(f1)

# ================= FINAL RESULTS =================
print("\n📊 FINAL METRICS:")
print(f"IoU       : {np.mean(iou_list):.4f}")
print(f"Dice      : {np.mean(dice_list):.4f}")
print(f"Precision : {np.mean(prec_list):.4f}")
print(f"Recall    : {np.mean(rec_list):.4f}")
print(f"F1 Score  : {np.mean(f1_list):.4f}")

In [ ]:
import torch
import numpy as np

model.eval()

def compute_metrics(pred, mask):
    pred = (pred > 0.5).astype(np.uint8)
    mask = (mask > 0.5).astype(np.uint8)

    TP = np.sum((pred == 1) & (mask == 1))
    TN = np.sum((pred == 0) & (mask == 0))
    FP = np.sum((pred == 1) & (mask == 0))
    FN = np.sum((pred == 0) & (mask == 1))

    # Avoid division errors
    eps = 1e-7

    iou = TP / (TP + FP + FN + eps)
    dice = (2 * TP) / (2 * TP + FP + FN + eps)
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)

    return iou, dice, precision, recall, f1

# ================= EVALUATION LOOP =================
iou_list, dice_list = [], []
prec_list, rec_list, f1_list = [], [], []

with torch.no_grad():
    for img, mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)
        pred = torch.sigmoid(pred).cpu().numpy()
        mask = mask.cpu().numpy()

        for p, m in zip(pred, mask):
            iou, dice, precision, recall, f1 = compute_metrics(p[0], m[0])

            iou_list.append(iou)
            dice_list.append(dice)
            prec_list.append(precision)
            rec_list.append(recall)
            f1_list.append(f1)

# ================= FINAL RESULTS =================
print("\n📊 FINAL METRICS:")
print(f"IoU       : {np.mean(iou_list):.4f}")
print(f"Dice      : {np.mean(dice_list):.4f}")
print(f"Precision : {np.mean(prec_list):.4f}")
print(f"Recall    : {np.mean(rec_list):.4f}")
print(f"F1 Score  : {np.mean(f1_list):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch
import numpy as np
import random
import rasterio

NUM_SAMPLES = 5
SIZE = 512

model.eval()

indices = random.sample(range(len(val_imgs)), NUM_SAMPLES)

for idx in indices:

    with rasterio.open(val_imgs[idx]) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    with rasterio.open(val_masks[idx]) as src:
        gt_mask = src.read(1)

    h, w = img.shape[:2]

    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    input_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred = model(input_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (w, h))
    pred_mask = (pred > 0.5).astype(np.uint8)

    # ===== METRICS =====
    iou, dice, precision, recall, f1 = compute_metrics(pred_mask, gt_mask)

    # ===== OVERLAY =====
    overlay = img.copy()
    overlay[pred_mask == 1] = [255, 0, 0]

    # ===== BOUNDARY =====
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary = img.copy()
    cv2.drawContours(boundary, contours, -1, (0,255,0), 2)

    # ===== PLOT =====
    plt.figure(figsize=(16,5))

    plt.subplot(1,5,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,5,2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,5,3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

    plt.subplot(1,5,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.subplot(1,5,5)
    plt.imshow(boundary)
    plt.title("Boundary")
    plt.axis("off")

    plt.suptitle(
        f"IoU:{iou:.3f} | Dice:{dice:.3f} | Prec:{precision:.3f} | Rec:{recall:.3f} | F1:{f1:.3f}",
        fontsize=12
    )

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2
import torch
import numpy as np
import random
import rasterio

NUM_SAMPLES = 5
SIZE = 512

model.eval()

indices = random.sample(range(len(val_imgs)), NUM_SAMPLES)

for idx in indices:

    with rasterio.open(val_imgs[idx]) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    with rasterio.open(val_masks[idx]) as src:
        gt_mask = src.read(1)

    h, w = img.shape[:2]

    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    input_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred = model(input_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (w, h))
    pred_mask = (pred > 0.5).astype(np.uint8)

    # ===== METRICS =====
    iou, dice, precision, recall, f1 = compute_metrics(pred_mask, gt_mask)

    # ===== OVERLAY =====
    overlay = img.copy()
    overlay[pred_mask == 1] = [255, 0, 0]

    # ===== BOUNDARY =====
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary = img.copy()
    cv2.drawContours(boundary, contours, -1, (0,255,0), 2)

    # ===== PLOT =====
    plt.figure(figsize=(16,5))

    plt.subplot(1,5,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,5,2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,5,3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

    plt.subplot(1,5,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.subplot(1,5,5)
    plt.imshow(boundary)
    plt.title("Boundary")
    plt.axis("off")

    plt.suptitle(
        f"IoU:{iou:.3f} | Dice:{dice:.3f} | Prec:{precision:.3f} | Rec:{recall:.3f} | F1:{f1:.3f}",
        fontsize=12
    )

    plt.show()

In [ ]:
# ================= INSTALL (if needed) =================
!pip install rasterio geopandas shapely fiona
!pip install segmentation-models-pytorch
!pip install segment-anything
!pip install timm
!pip install opencv-python

# ================= IMPORTS =================
import os
import cv2
import torch
import numpy as np
import warnings
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")  # 🔥 removes TIFF warnings

# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ================= DATASET =================
SIZE = 512

class TiffDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        # 🔥 Use rasterio instead of cv2 (fix TIFF warnings + geo data safe)
        with rasterio.open(self.imgs[idx]) as src:
            img = src.read([1,2,3]).transpose(1,2,0)

        img = cv2.resize(img, (SIZE, SIZE))
        img = img / 255.0

        with rasterio.open(self.masks[idx]) as src:
            mask = src.read(1)

        mask = cv2.resize(mask, (SIZE, SIZE))
        mask = (mask > 0).astype(np.float32)

        img = np.transpose(img, (2,0,1))
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img, dtype=torch.float32), \
               torch.tensor(mask, dtype=torch.float32)

# ================= PATHS =================
IMG_DIR = "//kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])

image_paths = [os.path.join(IMG_DIR, f) for f in images]
mask_paths  = [os.path.join(MASK_DIR, f) for f in images]

# ================= SPLIT =================
split = int(0.8 * len(image_paths))
train_imgs = image_paths[:split]
val_imgs   = image_paths[split:]

train_masks = mask_paths[:split]
val_masks   = mask_paths[split:]

print("Train:", len(train_imgs), "Val:", len(val_imgs))

# ================= DATALOADER =================
BATCH = 4

train_loader = DataLoader(
    TiffDataset(train_imgs, train_masks),
    batch_size=BATCH,
    shuffle=True,
    num_workers=0,   # 🔥 stable for TIFF
    pin_memory=True
)

val_loader = DataLoader(
    TiffDataset(val_imgs, val_masks),
    batch_size=BATCH,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)



In [ ]:
import segmentation_models_pytorch as smp

deeplab = smp.DeepLabV3Plus(
    encoder_name="resnet34",   # fast + stable
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

deeplab = torch.nn.DataParallel(deeplab)
deeplab = deeplab.to(device)

print("✅ DeepLabV3+ Ready")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ================= RESUME =================
CHECKPOINT = "/kaggle/working/deeplab_resume.pth"

if os.path.exists(CHECKPOINT):
    deeplab.module.load_state_dict(torch.load(CHECKPOINT))
    print("✅ Resumed DeepLab training")

# ================= LOSS =================
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(deeplab.parameters(), lr=1e-4)

scaler = torch.amp.GradScaler("cuda")

# ================= TRAIN =================
EPOCHS = 50
best_val = 1e9

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    deeplab.train()
    train_loss = 0

    for img, mask in train_loader:
        img = img.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            pred = deeplab(img)
            loss = loss_fn(pred, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    deeplab.eval()
    val_loss = 0

    with torch.no_grad():
        for img, mask in val_loader:
            img = img.to(device)
            mask = mask.to(device)

            with torch.amp.autocast("cuda"):
                pred = deeplab(img)
                loss = loss_fn(pred, mask)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # ===== SAVE =====
    torch.save(deeplab.module.state_dict(), CHECKPOINT)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(deeplab.module.state_dict(), "/kaggle/working/deeplab_best.pth")
        print("✅ DeepLab Best Saved")

print("🎉 DeepLab Training Done")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

# ================= CHECKPOINT PATH =================
CHECKPOINT = "/kaggle/working/deeplab_resume.pth"

# ================= LOSS =================
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(deeplab.parameters(), lr=1e-4)

scaler = torch.amp.GradScaler("cuda")

# ================= RESUME =================
start_epoch = 0
best_val = 1e9

if os.path.exists(CHECKPOINT):
    checkpoint = torch.load(CHECKPOINT)

    deeplab.module.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    best_val = checkpoint["best_val"]

    print(f"✅ Resumed from Epoch {start_epoch}")

# ================= TRAIN =================
EPOCHS = 50

for epoch in range(start_epoch, EPOCHS):

    # ===== TRAIN =====
    deeplab.train()
    train_loss = 0

    for img, mask in train_loader:
        img = img.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            pred = deeplab(img)
            loss = loss_fn(pred, mask)

        scaler.scale(loss).backward()

        # 🔥 prevent gradient explosion
        torch.nn.utils.clip_grad_norm_(deeplab.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    deeplab.eval()
    val_loss = 0

    with torch.no_grad():
        for img, mask in val_loader:
            img = img.to(device)
            mask = mask.to(device)

            with torch.amp.autocast("cuda"):
                pred = deeplab(img)
                loss = loss_fn(pred, mask)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # ===== SAVE FULL CHECKPOINT =====
    torch.save({
        "epoch": epoch,
        "model": deeplab.module.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "best_val": best_val
    }, CHECKPOINT)

    # ===== SAVE BEST MODEL =====
    if val_loss < best_val:
        best_val = val_loss
        torch.save(deeplab.module.state_dict(), "/kaggle/working/deeplab_best.pth")
        print("✅ DeepLab Best Saved")

    # ===== MEMORY FIX (IMPORTANT) =====
    torch.cuda.empty_cache()
    import gc
    gc.collect()

print("🎉 DeepLab Training Completed")

In [ ]:
deeplab.eval()

iou_list, dice_list = [], []
prec_list, rec_list, f1_list = [], [], []

with torch.no_grad():
    for img, mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = deeplab(img)
        pred = torch.sigmoid(pred).cpu().numpy()
        mask = mask.cpu().numpy()

        for p, m in zip(pred, mask):
            iou, dice, precision, recall, f1 = compute_metrics(p[0], m[0])

            iou_list.append(iou)
            dice_list.append(dice)
            prec_list.append(precision)
            rec_list.append(recall)
            f1_list.append(f1)

print("\n📊 DEEPLAB METRICS:")
print(f"IoU       : {np.mean(iou_list):.4f}")
print(f"Dice      : {np.mean(dice_list):.4f}")
print(f"Precision : {np.mean(prec_list):.4f}")
print(f"Recall    : {np.mean(rec_list):.4f}")
print(f"F1 Score  : {np.mean(f1_list):.4f}")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random
import rasterio

NUM_SAMPLES = 5
SIZE = 512

deeplab.eval()

indices = random.sample(range(len(val_imgs)), NUM_SAMPLES)

for idx in indices:

    with rasterio.open(val_imgs[idx]) as src:
        img = src.read([1,2,3]).transpose(1,2,0)

    with rasterio.open(val_masks[idx]) as src:
        gt_mask = src.read(1)

    h, w = img.shape[:2]

    img_resized = cv2.resize(img, (SIZE, SIZE)) / 255.0
    input_tensor = torch.tensor(img_resized.transpose(2,0,1)).unsqueeze(0).float().to(device)

    with torch.no_grad():
        pred = deeplab(input_tensor)
        pred = torch.sigmoid(pred).cpu().numpy()[0,0]

    pred = cv2.resize(pred, (w, h))
    pred_mask = (pred > 0.5).astype(np.uint8)

    # ===== METRICS =====
    iou, dice, precision, recall, f1 = compute_metrics(pred_mask, gt_mask)

    # ===== OVERLAY =====
    overlay = img.copy()
    overlay[pred_mask == 1] = [255, 0, 0]

    # ===== BOUNDARY =====
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary = img.copy()
    cv2.drawContours(boundary, contours, -1, (0,255,0), 2)

    # ===== PLOT =====
    plt.figure(figsize=(16,5))

    plt.subplot(1,5,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,5,2)
    plt.imshow(gt_mask, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,5,3)
    plt.imshow(pred_mask, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")

    plt.subplot(1,5,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.subplot(1,5,5)
    plt.imshow(boundary)
    plt.title("Boundary")
    plt.axis("off")

    plt.suptitle(
        f"DeepLab → IoU:{iou:.3f} Dice:{dice:.3f} F1:{f1:.3f}",
        fontsize=12
    )

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ===== UNET =====
plt.figure()
plt.plot(unet_train_hist, label="UNet Train Loss")
plt.plot(unet_val_hist, label="UNet Val Loss")
plt.title("UNet Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

# ===== DEEPLAB =====
plt.figure()
plt.plot(deeplab_train_hist, label="DeepLab Train Loss")
plt.plot(deeplab_val_hist, label="DeepLab Val Loss")
plt.title("DeepLabV3+ Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure()

plt.plot(unet_val_hist, label="UNet Val Loss")
plt.plot(deeplab_val_hist, label="DeepLab Val Loss")

plt.title("UNet vs DeepLab (Validation Loss Comparison)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.show()

In [ ]:
import numpy as np

# ===== FINAL VALUES (replace with your results) =====
unet_scores = [np.mean(iou_list), np.mean(dice_list), np.mean(f1_list)]
deeplab_scores = [np.mean(iou_list), np.mean(dice_list), np.mean(f1_list)]  # replace with deeplab values

labels = ["IoU", "Dice", "F1"]

x = np.arange(len(labels))
width = 0.35

plt.figure()

plt.bar(x - width/2, unet_scores, width, label="UNet")
plt.bar(x + width/2, deeplab_scores, width, label="DeepLab")

plt.xticks(x, labels)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.legend()
plt.grid()

plt.show()

In [ ]:
metrics = ["Precision", "Recall", "F1"]

unet_vals = [np.mean(prec_list), np.mean(rec_list), np.mean(f1_list)]
deeplab_vals = [np.mean(prec_list), np.mean(rec_list), np.mean(f1_list)]  # replace

x = np.arange(len(metrics))

plt.figure()

plt.bar(x - 0.2, unet_vals, 0.4, label="UNet")
plt.bar(x + 0.2, deeplab_vals, 0.4, label="DeepLab")

plt.xticks(x, metrics)
plt.title("Precision / Recall / F1 Comparison")
plt.legend()
plt.grid()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TP, TN, FP, FN = 0, 0, 0, 0

model.eval()

with torch.no_grad():
    for img, mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)
        pred = torch.sigmoid(pred).cpu().numpy()
        mask = mask.cpu().numpy()

        pred = (pred > 0.5).astype(np.uint8)

        TP += np.sum((pred == 1) & (mask == 1))
        TN += np.sum((pred == 0) & (mask == 0))
        FP += np.sum((pred == 1) & (mask == 0))
        FN += np.sum((pred == 0) & (mask == 1))

# ===== MATRIX =====
cm = np.array([[TP, FP],
               [FN, TN]])

labels = [["TP", "FP"], ["FN", "TN"]]

plt.figure()

plt.imshow(cm)
plt.title("Confusion Matrix (Pixel-wise)")
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{labels[i][j]}\n{cm[i,j]}", ha='center', va='center')

plt.xticks([0,1], ["Pred: Building", "Pred: Background"])
plt.yticks([0,1], ["Actual: Building", "Actual: Background"])

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TP, TN, FP, FN = 0, 0, 0, 0

model.eval()

with torch.no_grad():
    for img, mask in val_loader:

        img = img.to(device)
        mask = mask.to(device)

        pred = model(img)
        pred = torch.sigmoid(pred).cpu().numpy()
        mask = mask.cpu().numpy()

        pred = (pred > 0.5).astype(np.uint8)

        TP += np.sum((pred == 1) & (mask == 1))
        TN += np.sum((pred == 0) & (mask == 0))
        FP += np.sum((pred == 1) & (mask == 0))
        FN += np.sum((pred == 0) & (mask == 1))

# ===== MATRIX =====
cm = np.array([[TP, FP],
               [FN, TN]])

labels = [["TP", "FP"], ["FN", "TN"]]

plt.figure()

plt.imshow(cm)
plt.title("Confusion Matrix (Pixel-wise)")
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{labels[i][j]}\n{cm[i,j]}", ha='center', va='center')

plt.xticks([0,1], ["Pred: Building", "Pred: Background"])
plt.yticks([0,1], ["Actual: Building", "Actual: Background"])

plt.show()

In [ ]:
# ================= INSTALL =================
!pip install -q segment-anything rasterio shapely geopandas

# ================= IMPORTS =================
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

from segment_anything import sam_model_registry, SamPredictor

import rasterio
import geopandas as gpd
from shapely.geometry import Polygon

# ================= DEVICE =================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ================= DATASET =================
SIZE = 512

class TiffDataset(Dataset):
    def __init__(self, imgs, masks):
        self.imgs = imgs
        self.masks = masks

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx])
        img = cv2.resize(img, (SIZE, SIZE))

        mask = cv2.imread(self.masks[idx], 0)
        mask = cv2.resize(mask, (SIZE, SIZE))
        mask = (mask > 127).astype(np.uint8)

        return img, mask, self.imgs[idx]

# ================= PATHS =================
IMG_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])
img_paths = [os.path.join(IMG_DIR, f) for f in images]
mask_paths = [os.path.join(MASK_DIR, f) for f in images]

dataset = TiffDataset(img_paths, mask_paths)
loader = DataLoader(dataset, batch_size=1, shuffle=False)

# ================= LOAD SAM =================
sam = sam_model_registry["vit_b"](checkpoint="/kaggle/input/sam-vit/sam_vit_b.pth")
sam.to(device)

predictor = SamPredictor(sam)

# ================= SAVE DIR =================
SAVE_DIR = "/kaggle/working/sam_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================= FUNCTION =================
def save_shapefile(mask, img_path, save_path):
    with rasterio.open(img_path) as src:
        transform = src.transform

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    for cnt in contours:
        coords = []
        for p in cnt:
            x, y = p[0]
            coords.append((x, y))
        if len(coords) > 3:
            polygons.append(Polygon(coords))

    gdf = gpd.GeoDataFrame(geometry=polygons)
    gdf.to_file(save_path)

# ================= INFERENCE =================
for i, (img, gt_mask, path) in enumerate(loader):

    img = img[0].numpy()
    gt_mask = gt_mask[0].numpy()

    predictor.set_image(img)

    h, w, _ = img.shape

    # 🔥 automatic grid prompts
    input_points = np.array([[w//2, h//2]])
    input_labels = np.array([1])

    masks, scores, _ = predictor.predict(
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=True
    )

    pred_mask = masks[0].astype(np.uint8) * 255

    # ================= SAVE =================
    name = os.path.basename(path[0]).replace(".tif", "")

    cv2.imwrite(f"{SAVE_DIR}/{name}_mask.tif", pred_mask)
    cv2.imwrite(f"{SAVE_DIR}/{name}_mask.jpg", pred_mask)

    # ================= OVERLAY =================
    overlay = img.copy()
    overlay[pred_mask > 0] = [0, 0, 255]

    cv2.imwrite(f"{SAVE_DIR}/{name}_overlay.jpg", overlay)

    # ================= BOUNDARY =================
    edges = cv2.Canny(pred_mask, 100, 200)
    cv2.imwrite(f"{SAVE_DIR}/{name}_boundary.jpg", edges)

    # ================= SHP =================
    save_shapefile(pred_mask, path[0], f"{SAVE_DIR}/{name}.shp")

    # ================= VISUAL =================
    if i < 5:
        plt.figure(figsize=(12,6))

        plt.subplot(1,5,1)
        plt.imshow(img)
        plt.title("Image")

        plt.subplot(1,5,2)
        plt.imshow(gt_mask, cmap="gray")
        plt.title("GT Mask")

        plt.subplot(1,5,3)
        plt.imshow(pred_mask, cmap="gray")
        plt.title("Prediction")

        plt.subplot(1,5,4)
        plt.imshow(overlay)
        plt.title("Overlay")

        plt.subplot(1,5,5)
        plt.imshow(edges, cmap="gray")
        plt.title("Boundary")

        plt.show()

print("✅ SAM Processing Completed")

In [ ]:
# ================= IMPORTS =================
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= LOAD MODELS =================
unet.load_state_dict(torch.load("/kaggle/working/output/unet_best.pth"))
deeplab.load_state_dict(torch.load("/kaggle/working/deeplab_best.pth"))

unet.eval()
deeplab.eval()

# ================= SAMPLE SELECTION =================
indices = random.sample(range(len(val_dataset)), 5)

ious_unet, ious_deep = [], []

# ================= FUNCTION =================
def compute_iou(pred, mask):
    pred = (pred > 0.5).astype(np.uint8)
    mask = mask.astype(np.uint8)

    intersection = np.logical_and(pred, mask).sum()
    union = np.logical_or(pred, mask).sum()

    return intersection / (union + 1e-6)

# ================= LOOP =================
for idx in indices:

    img, mask = val_dataset[idx]

    img_np = np.transpose(img.numpy(), (1,2,0))
    mask_np = mask.numpy()[0]

    img_tensor = img.unsqueeze(0).to(device)

    # ===== UNET =====
    with torch.no_grad():
        pred_unet = torch.sigmoid(unet(img_tensor)).cpu().numpy()[0,0]

    # ===== DEEPLAB =====
    with torch.no_grad():
        pred_deep = torch.sigmoid(deeplab(img_tensor)).cpu().numpy()[0,0]

    # ===== RF (simple simulation if already trained) =====
    rf_pred = (pred_unet > 0.5).astype(np.uint8)  # placeholder

    # ===== SAM (use mask-like prediction) =====
    sam_pred = (pred_deep > 0.5).astype(np.uint8)  # placeholder

    # ===== IOU =====
    iou_u = compute_iou(pred_unet, mask_np)
    iou_d = compute_iou(pred_deep, mask_np)

    ious_unet.append(iou_u)
    ious_deep.append(iou_d)

    # ===== OVERLAY FUNCTION =====
    def overlay(img, pred):
        color = np.zeros_like(img)
        color[:,:,1] = 255   # green
        color[:,:,2] = 255   # blue → CYAN

        mask = pred > 0.5
        img_overlay = img.copy()
        img_overlay[mask] = img_overlay[mask]*0.5 + color[mask]*0.5
        return img_overlay

    # ===== BOUNDARY =====
    def get_boundary(pred):
        pred = (pred > 0.5).astype(np.uint8)
        edges = cv2.Canny(pred*255, 100, 200)
        return edges

    # ================= PLOT =================
    fig, ax = plt.subplots(2,5, figsize=(20,8))

    ax[0,0].imshow(img_np)
    ax[0,0].set_title("Original")

    ax[0,1].imshow(mask_np, cmap='gray')
    ax[0,1].set_title("GT Mask")

    ax[0,2].imshow(pred_unet, cmap='gray')
    ax[0,2].set_title(f"UNet (IoU {iou_u:.2f})")

    ax[0,3].imshow(pred_deep, cmap='gray')
    ax[0,3].set_title(f"DeepLab (IoU {iou_d:.2f})")

    ax[0,4].imshow(rf_pred, cmap='gray')
    ax[0,4].set_title("RF")

    # second row
    ax[1,0].imshow(overlay(img_np, pred_unet))
    ax[1,0].set_title("UNet Overlay")

    ax[1,1].imshow(overlay(img_np, pred_deep))
    ax[1,1].set_title("DeepLab Overlay")

    ax[1,2].imshow(get_boundary(pred_unet), cmap='gray')
    ax[1,2].set_title("UNet Boundary")

    ax[1,3].imshow(get_boundary(pred_deep), cmap='gray')
    ax[1,3].set_title("DeepLab Boundary")

    ax[1,4].imshow(sam_pred, cmap='gray')
    ax[1,4].set_title("SAM")

    for a in ax.flatten():
        a.axis("off")

    plt.show()

# ================= METRICS =================
print("📊 Average IoU")
print("UNet   :", np.mean(ious_unet))
print("DeepLab:", np.mean(ious_deep))

# ================= GRAPH =================
plt.figure(figsize=(8,5))
plt.plot(ious_unet, label="UNet IoU")
plt.plot(ious_deep, label="DeepLab IoU")
plt.legend()
plt.title("IoU Comparison")
plt.xlabel("Samples")
plt.ylabel("IoU")
plt.show()

# ================= CONFUSION MATRIX (UNET) =================
y_true = []
y_pred = []

for idx in indices:
    img, mask = val_dataset[idx]
    img_tensor = img.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.sigmoid(unet(img_tensor)).cpu().numpy()[0,0]

    pred = (pred > 0.5).flatten()
    mask = mask.numpy().flatten()

    y_true.extend(mask)
    y_pred.extend(pred)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (UNet)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# ================= IMPORTS =================
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= MODEL PATHS =================
UNET_PATH = "/kaggle/working/unet_best.pth"
DEEPLAB_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth"

# ================= LOAD MODELS =================
unet = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

deeplab = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

unet.load_state_dict(torch.load(UNET_PATH, map_location=device))
deeplab.load_state_dict(torch.load(DEEPLAB_PATH, map_location=device))

unet.eval()
deeplab.eval()

print("✅ Models Loaded")

# ================= HELPER FUNCTIONS =================
def predict(model, img):
    with torch.no_grad():
        img = torch.tensor(img).unsqueeze(0).to(device)
        pred = model(img)
        pred = torch.sigmoid(pred)
        pred = (pred > 0.5).float()
    return pred.cpu().numpy()[0,0]

def overlay(img, mask):
    img = img.copy()
    mask = mask.astype(bool)
    img[mask] = [0, 255, 255]   # cyan overlay
    return img

def get_boundary(mask):
    mask = (mask > 0).astype(np.uint8)
    edges = cv2.Canny(mask*255, 100, 200)
    return edges

def compute_iou(pred, gt):
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-6)

# ================= VISUALIZATION =================
samples = 5

ious_unet = []
ious_deeplab = []

for i in range(samples):

    img = cv2.imread(val_imgs[i])
    img = cv2.resize(img, (512,512))
    img_norm = img / 255.0
    img_tensor = np.transpose(img_norm, (2,0,1)).astype(np.float32)

    mask = cv2.imread(val_masks[i], 0)
    mask = cv2.resize(mask, (512,512))
    mask = (mask > 127).astype(np.float32)

    # predictions
    pred_unet = predict(unet, img_tensor)
    pred_deeplab = predict(deeplab, img_tensor)

    # IoU
    ious_unet.append(compute_iou(pred_unet, mask))
    ious_deeplab.append(compute_iou(pred_deeplab, mask))

    # overlays
    overlay_unet = overlay(img.copy(), pred_unet)
    overlay_deeplab = overlay(img.copy(), pred_deeplab)

    # boundaries
    boundary_unet = get_boundary(pred_unet)
    boundary_deeplab = get_boundary(pred_deeplab)

    # ================= PLOT =================
    plt.figure(figsize=(18,8))

    plt.subplot(2,5,1); plt.imshow(img[:,:,::-1]); plt.title("Original"); plt.axis("off")
    plt.subplot(2,5,2); plt.imshow(mask, cmap='gray'); plt.title("GT Mask"); plt.axis("off")
    plt.subplot(2,5,3); plt.imshow(pred_unet, cmap='gray'); plt.title("UNet"); plt.axis("off")
    plt.subplot(2,5,4); plt.imshow(overlay_unet[:,:,::-1]); plt.title("UNet Overlay"); plt.axis("off")
    plt.subplot(2,5,5); plt.imshow(boundary_unet, cmap='gray'); plt.title("UNet Boundary"); plt.axis("off")

    plt.subplot(2,5,6); plt.imshow(img[:,:,::-1]); plt.title("Original"); plt.axis("off")
    plt.subplot(2,5,7); plt.imshow(mask, cmap='gray'); plt.title("GT Mask"); plt.axis("off")
    plt.subplot(2,5,8); plt.imshow(pred_deeplab, cmap='gray'); plt.title("DeepLab"); plt.axis("off")
    plt.subplot(2,5,9); plt.imshow(overlay_deeplab[:,:,::-1]); plt.title("DeepLab Overlay"); plt.axis("off")
    plt.subplot(2,5,10); plt.imshow(boundary_deeplab, cmap='gray'); plt.title("DeepLab Boundary"); plt.axis("off")

    plt.show()

# ================= METRICS =================
print("\n📊 IoU Scores")
for i in range(samples):
    print(f"Sample {i+1}: UNet={ious_unet[i]:.4f}, DeepLab={ious_deeplab[i]:.4f}")

print("\n📈 Average IoU")
print("UNet:", np.mean(ious_unet))
print("DeepLab:", np.mean(ious_deeplab))

# ================= GRAPH =================
plt.figure()
plt.plot(ious_unet, label="UNet IoU")
plt.plot(ious_deeplab, label="DeepLab IoU")
plt.legend()
plt.title("IoU Comparison")
plt.xlabel("Samples")
plt.ylabel("IoU")
plt.show()

# ================= CONFUSION MATRIX =================
all_preds = []
all_gt = []

for i in range(samples):
    mask = cv2.imread(val_masks[i], 0)
    mask = cv2.resize(mask, (512,512))
    mask = (mask > 127).astype(np.int32)

    pred = predict(unet, img_tensor)
    pred = pred.astype(np.int32)

    all_preds.extend(pred.flatten())
    all_gt.extend(mask.flatten())

cm = confusion_matrix(all_gt, all_preds)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix (UNet)")
plt.colorbar()
plt.show()

In [ ]:
# ================= IMPORTS =================
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= PATHS =================
UNET_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth"
DEEPLAB_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth"

IMG_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

# ================= LOAD IMAGE PATHS =================
images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])[:5]

# ================= LOAD MODELS =================
unet = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)
deeplab = smp.DeepLabV3Plus("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)

def load_state_dict_strip_module(model, path, device):
    state_dict = torch.load(path, map_location=device)
    new_state_dict = {
        k[7:] if k.startswith("module.") else k: v
        for k, v in state_dict.items()
    }
    model.load_state_dict(new_state_dict)
    return model

unet = load_state_dict_strip_module(unet, UNET_PATH, device)
deeplab = load_state_dict_strip_module(deeplab, DEEPLAB_PATH, device)

unet.eval()
deeplab.eval()

print("✅ Models Loaded")
print(f"✅ Found {len(images)} images: {images}")

In [ ]:
# ================= IMPORTS =================
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================= PATHS =================
UNET_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/unet_best (1).pth"
DEEPLAB_PATH = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1/deeplab_best.pth"

IMG_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

# ================= LOAD MODELS =================
unet = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)
deeplab = smp.DeepLabV3Plus("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)  # ← was resnet50

def load_state_dict_strip_module(model, path, device):
    state_dict = torch.load(path, map_location=device)
    new_state_dict = {
        k[7:] if k.startswith("module.") else k: v
        for k, v in state_dict.items()
    }
    model.load_state_dict(new_state_dict)
    return model

unet = load_state_dict_strip_module(unet, UNET_PATH, device)
deeplab = load_state_dict_strip_module(deeplab, DEEPLAB_PATH, device)

unet.eval()
deeplab.eval()

print("✅ Models Loaded")
# ================= METRICS =================
def compute_metrics(pred, gt):
    pred = pred.flatten()
    gt = gt.flatten()

    intersection = np.sum(pred * gt)
    union = np.sum(pred) + np.sum(gt) - intersection

    iou = intersection / (union + 1e-6)
    dice = (2 * intersection) / (np.sum(pred) + np.sum(gt) + 1e-6)
    acc = np.mean(pred == gt)

    return iou, dice, acc

# ================= VISUALIZATION =================
ious_unet, ious_deep = [], []

plt.figure(figsize=(20, 25))

for i, name in enumerate(images):

    # ----- LOAD IMAGE -----
    img_path = os.path.join(IMG_DIR, name)
    mask_path = os.path.join(MASK_DIR, name)

    img = cv2.imread(img_path)
    img = cv2.resize(img, (512, 512))
    img_norm = img / 255.0

    mask = cv2.imread(mask_path, 0)
    mask = cv2.resize(mask, (512, 512))
    mask = (mask > 127).astype(np.uint8)

    # ----- PREPARE TENSOR -----
    x = np.transpose(img_norm, (2,0,1))
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

    # ----- PREDICT -----
    with torch.no_grad():
        p_unet = torch.sigmoid(unet(x))[0,0].cpu().numpy()
        p_deep = torch.sigmoid(deeplab(x))[0,0].cpu().numpy()

    p_unet = (p_unet > 0.5).astype(np.uint8)
    p_deep = (p_deep > 0.5).astype(np.uint8)

    # ----- METRICS -----
    iou_u, dice_u, acc_u = compute_metrics(p_unet, mask)
    iou_d, dice_d, acc_d = compute_metrics(p_deep, mask)

    ious_unet.append(iou_u)
    ious_deep.append(iou_d)

    # ----- OVERLAY -----
    overlay = img.copy()
    overlay[p_unet==1] = [255,255,0]  # light cyan

    # ----- BOUNDARY -----
    edges = cv2.Canny(p_unet*255, 100, 200)

    # ================= PLOT =================
    row = i*5

    plt.subplot(5,5,row+1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')

    plt.subplot(5,5,row+2)
    plt.imshow(mask, cmap='gray')
    plt.title("GT Mask")
    plt.axis('off')

    plt.subplot(5,5,row+3)
    plt.imshow(p_unet, cmap='gray')
    plt.title(f"UNet\nIoU:{iou_u:.2f}")
    plt.axis('off')

    plt.subplot(5,5,row+4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis('off')

    plt.subplot(5,5,row+5)
    plt.imshow(edges, cmap='gray')
    plt.title("Boundary")
    plt.axis('off')

plt.tight_layout()
plt.show()

# ================= IOU GRAPH =================
plt.figure(figsize=(8,5))
plt.plot(ious_unet, label="UNet")
plt.plot(ious_deep, label="DeepLab")
plt.title("IoU Comparison")
plt.legend()
plt.show()

# ================= CONFUSION MATRIX =================
all_preds = []
all_gt = []

for name in images:
    mask = cv2.imread(os.path.join(MASK_DIR, name), 0)
    mask = cv2.resize(mask, (512,512))
    mask = (mask>127).astype(np.uint8)

    all_gt.extend(mask.flatten())
    all_preds.extend(p_unet.flatten())

cm = confusion_matrix(all_gt, all_preds)

plt.imshow(cm, cmap='Blues')
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

print("🎉 DONE: Visualization + Metrics + Graphs")

In [ ]:
# ============================================================
# CELL 0 — IMPORTS & SETUP
# ============================================================
import os, cv2, torch, pickle, numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix, accuracy_score, jaccard_score, f1_score
import matplotlib.patches as mpatches

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

BASE = "/kaggle/input/models/pravallika416/all-models-for-building-footprint-extraction/pytorch/default/1"
IMG_DIR  = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/images"
MASK_DIR = "/kaggle/input/datasets/huanranye/inria-aerial-image-labeling-dataset/train/ground_truth"

images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])[:5]
print(f"Found {len(images)} images: {images}")

# ---- helpers ----
def load_strip(model, path):
    sd = torch.load(path, map_location=device)
    sd = {k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}
    model.load_state_dict(sd)
    model.eval()
    return model

def load_image(name):
    img  = cv2.imread(os.path.join(IMG_DIR,  name))
    img  = cv2.resize(img, (512, 512))
    mask = cv2.imread(os.path.join(MASK_DIR, name), 0)
    mask = cv2.resize(mask, (512, 512))
    mask = (mask > 127).astype(np.uint8)
    return img, mask

def to_tensor(img):
    x = np.transpose(img / 255.0, (2, 0, 1))
    return torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

def compute_metrics(pred, gt):
    p, g = pred.flatten(), gt.flatten()
    iou   = jaccard_score(g, p, zero_division=0)
    dice  = f1_score(g, p, zero_division=0)
    acc   = accuracy_score(g, p)
    tp = np.sum((p==1)&(g==1)); fp = np.sum((p==1)&(g==0))
    fn = np.sum((p==0)&(g==1)); tn = np.sum((p==0)&(g==0))
    prec = tp/(tp+fp+1e-6); rec = tp/(tp+fn+1e-6)
    return dict(IoU=iou, Dice=dice, Acc=acc, Precision=prec, Recall=rec, CM=confusion_matrix(g,p))

def show_metrics(m, name):
    print(f"\n{'='*45}")
    print(f"  {name} — Aggregate Metrics (5 images)")
    print(f"{'='*45}")
    for k, v in m.items():
        if k != "CM": print(f"  {k:<12}: {v:.4f}")
    print(f"\n  Confusion Matrix:\n{m['CM']}")
    print(f"{'='*45}\n")

In [ ]:
# ============================================================
# CELL 1 — UNet (ResNet-34)
# ============================================================
unet = smp.Unet("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)
unet = load_strip(unet, f"{BASE}/unet_best (1).pth")
print("✅ UNet loaded")

all_pred, all_gt = [], []
per_image_ious = []  # store as we go — no need to re-run inference

fig, axes = plt.subplots(5, 5, figsize=(20, 20))
fig.suptitle("UNet — Inference Results", fontsize=16, fontweight='bold', y=1.01)

col_titles = ["Original", "Ground Truth", "Prediction", "Overlay", "Boundary"]
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=12, fontweight='bold')

for i, name in enumerate(images):
    img, mask = load_image(name)

    with torch.no_grad():
        pred = torch.sigmoid(unet(to_tensor(img)))[0, 0].detach().cpu().numpy()  # ← .detach() fix
    pred_bin = (pred > 0.5).astype(np.uint8)

    m = compute_metrics(pred_bin, mask)
    per_image_ious.append(m['IoU'])           # ← reuse, no second inference
    all_pred.extend(pred_bin.flatten())
    all_gt.extend(mask.flatten())

    overlay = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    overlay[pred_bin == 1] = [255, 255, 0]
    edges = cv2.Canny(pred_bin * 255, 100, 200)

    axes[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_ylabel(name[:12], fontsize=8)
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 2].imshow(pred_bin, cmap='gray')
    axes[i, 2].set_xlabel(f"IoU:{m['IoU']:.2f} Dice:{m['Dice']:.2f}", fontsize=8)
    axes[i, 3].imshow(overlay)
    axes[i, 4].imshow(edges, cmap='gray')

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.savefig("unet_results.png", dpi=150, bbox_inches='tight')
plt.show()

# Aggregate metrics
agg = compute_metrics(np.array(all_pred), np.array(all_gt))
show_metrics(agg, "UNet")

# IoU bar chart — uses already-collected per_image_ious
plt.figure(figsize=(8, 4))
bars = plt.bar(range(len(images)), per_image_ious, color='steelblue', edgecolor='black')
plt.xticks(range(len(images)), [n[:10] for n in images], rotation=15)
plt.ylim(0, 1)
plt.ylabel("IoU")
plt.title("UNet — Per-Image IoU")
for b, v in zip(bars, per_image_ious):
    plt.text(b.get_x() + 0.1, v + 0.01, f"{v:.2f}", fontsize=9)
plt.tight_layout()
plt.savefig("unet_iou.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 2 — DeepLabV3+ (ResNet-34)
# ============================================================
deeplab = smp.DeepLabV3Plus("resnet34", encoder_weights=None, in_channels=3, classes=1).to(device)
deeplab = load_strip(deeplab, f"{BASE}/deeplab_best.pth")
print("✅ DeepLabV3+ loaded")

all_pred, all_gt = [], []
fig, axes = plt.subplots(5, 5, figsize=(20, 20))
fig.suptitle("DeepLabV3+ — Inference Results", fontsize=16, fontweight='bold', y=1.01)

col_titles = ["Original", "Ground Truth", "Prediction", "Overlay", "Boundary"]
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=12, fontweight='bold')

for i, name in enumerate(images):
    img, mask = load_image(name)

    with torch.no_grad():
        pred = torch.sigmoid(deeplab(to_tensor(img)))[0,0].cpu().numpy()
    pred_bin = (pred > 0.5).astype(np.uint8)

    m = compute_metrics(pred_bin, mask)
    all_pred.extend(pred_bin.flatten()); all_gt.extend(mask.flatten())

    overlay = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    overlay[pred_bin==1] = [0, 255, 128]   # green tint for DeepLab
    edges = cv2.Canny(pred_bin*255, 100, 200)

    axes[i,0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i,0].set_ylabel(name[:12], fontsize=8)
    axes[i,1].imshow(mask, cmap='gray')
    axes[i,2].imshow(pred_bin, cmap='gray')
    axes[i,2].set_xlabel(f"IoU:{m['IoU']:.2f} Dice:{m['Dice']:.2f}", fontsize=8)
    axes[i,3].imshow(overlay)
    axes[i,4].imshow(edges, cmap='gray')

    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig("deeplab_results.png", dpi=150, bbox_inches='tight')
plt.show()

agg = compute_metrics(np.array(all_pred), np.array(all_gt))
show_metrics(agg, "DeepLabV3+")

ious = [compute_metrics(
            (torch.sigmoid(deeplab(to_tensor(load_image(n)[0]))).squeeze().cpu().numpy() > 0.5).astype(np.uint8),
            load_image(n)[1])['IoU'] for n in images]

plt.figure(figsize=(8,4))
bars = plt.bar(range(len(images)), ious, color='seagreen', edgecolor='black')
plt.xticks(range(len(images)), [n[:10] for n in images], rotation=15)
plt.ylim(0,1); plt.ylabel("IoU"); plt.title("DeepLabV3+ — Per-Image IoU")
for b, v in zip(bars, ious): plt.text(b.get_x()+0.1, v+0.01, f"{v:.2f}", fontsize=9)
plt.tight_layout(); plt.savefig("deeplab_iou.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

# # Optional: suppress TIFF warnings
# cv2.utils.logging.setLogLevel(cv2.utils.logging.ERROR)

# ============================================================
# MODEL LOAD
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deeplab = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
).to(device)

deeplab = load_strip(deeplab, f"{BASE}/deeplab_best.pth")
deeplab.eval()

print("✅ DeepLabV3+ loaded")

# ============================================================
# STORAGE
# ============================================================
all_pred, all_gt = [], []

fig, axes = plt.subplots(5, 5, figsize=(20, 20))
fig.suptitle("DeepLabV3+ — Inference Results", fontsize=16, fontweight='bold', y=1.01)

col_titles = ["Original", "Ground Truth", "Prediction", "Overlay", "Boundary"]
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=12, fontweight='bold')

# ============================================================
# INFERENCE LOOP
# ============================================================
with torch.no_grad():
    for i, name in enumerate(images):
        img, mask = load_image(name)

        # Prepare tensor
        tensor_img = to_tensor(img).to(device)

        # Prediction
        pred = torch.sigmoid(deeplab(tensor_img))[0, 0].cpu().numpy()
        pred_bin = (pred > 0.5).astype(np.uint8)

        # Metrics
        m = compute_metrics(pred_bin, mask)
        all_pred.extend(pred_bin.flatten())
        all_gt.extend(mask.flatten())

        # Visualization
        overlay = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
        overlay[pred_bin == 1] = [0, 255, 128]  # green tint
        edges = cv2.Canny(pred_bin * 255, 100, 200)

        axes[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i, 0].set_ylabel(name[:12], fontsize=8)

        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 2].imshow(pred_bin, cmap='gray')
        axes[i, 2].set_xlabel(f"IoU:{m['IoU']:.2f} Dice:{m['Dice']:.2f}", fontsize=8)

        axes[i, 3].imshow(overlay)
        axes[i, 4].imshow(edges, cmap='gray')

        for ax in axes[i]:
            ax.axis('off')

# ============================================================
# SAVE GRID
# ============================================================
plt.tight_layout()
plt.savefig("deeplab_results.png", dpi=150, bbox_inches='tight')
plt.show()

# ============================================================
# AGGREGATE METRICS
# ============================================================
agg = compute_metrics(np.array(all_pred), np.array(all_gt))
show_metrics(agg, "DeepLabV3+")

# ============================================================
# PER-IMAGE IoU
# ============================================================
ious = []

with torch.no_grad():
    for n in images:
        img, mask = load_image(n)
        pred = torch.sigmoid(deeplab(to_tensor(img).to(device)))[0, 0].cpu().numpy()
        pred_bin = (pred > 0.5).astype(np.uint8)
        ious.append(compute_metrics(pred_bin, mask)['IoU'])

# ============================================================
# IoU BAR PLOT
# ============================================================
plt.figure(figsize=(8, 4))
bars = plt.bar(range(len(images)), ious, color='seagreen', edgecolor='black')

plt.xticks(range(len(images)), [n[:10] for n in images], rotation=15)
plt.ylim(0, 1)
plt.ylabel("IoU")
plt.title("DeepLabV3+ — Per-Image IoU")

for b, v in zip(bars, ious):
    plt.text(b.get_x() + 0.1, v + 0.01, f"{v:.2f}", fontsize=9)

plt.tight_layout()
plt.savefig("deeplab_iou.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 3 — Random Forest
# ============================================================
with open(f"{BASE}/rf_model.pkl", "rb") as f:
    rf = pickle.load(f)
print("✅ Random Forest loaded")

def extract_features(img):
    """Per-pixel features: RGB + Sobel edges + HSV"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sobelx**2 + sobely**2)
    feats = np.stack([
        img[:,:,0], img[:,:,1], img[:,:,2],
        hsv[:,:,0], hsv[:,:,1], hsv[:,:,2],
        mag
    ], axis=-1).reshape(-1, 7) / 255.0
    return feats

all_pred, all_gt = [], []
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
fig.suptitle("Random Forest — Inference Results", fontsize=16, fontweight='bold', y=1.01)

for ax, t in zip(axes[0], ["Original", "Ground Truth", "Prediction", "Overlay"]):
    ax.set_title(t, fontsize=12, fontweight='bold')

for i, name in enumerate(images):
    img, mask = load_image(name)
    feats = extract_features(img)
    pred_bin = rf.predict(feats).reshape(512, 512).astype(np.uint8)

    m = compute_metrics(pred_bin, mask)
    all_pred.extend(pred_bin.flatten()); all_gt.extend(mask.flatten())

    overlay = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    overlay[pred_bin==1] = [255, 100, 100]  # red tint for RF

    axes[i,0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i,0].set_ylabel(name[:12], fontsize=8)
    axes[i,1].imshow(mask, cmap='gray')
    axes[i,2].imshow(pred_bin, cmap='gray')
    axes[i,2].set_xlabel(f"IoU:{m['IoU']:.2f} Dice:{m['Dice']:.2f}", fontsize=8)
    axes[i,3].imshow(overlay)

    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig("rf_results.png", dpi=150, bbox_inches='tight')
plt.show()

agg = compute_metrics(np.array(all_pred), np.array(all_gt))
show_metrics(agg, "Random Forest")